# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent handles:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- Word/sentence statistics → Text Stats Tool *(bonus)*
- Current date/time → DateTime Tool *(bonus)*
- Unit conversion (length & weight) → Unit Converter Tool *(bonus)*
- Sentiment of a piece of text → Sentiment Analyzer Tool *(bonus)*
- General queries → Direct response

---
### 🛠️ What's Implemented
- Agent logic with a central `agent()` entry point
- **Regex-based** conditional routing (not just plain substring matching, so `"80 + 90"` is
  correctly routed to the calculator even without the word "calculate")
- Tool integration (6 tools)
- Robust error handling at both the tool level and the agent level
- A **safe** calculator that parses expressions with `ast` instead of calling raw `eval()`

### 🚀 Bonus Features Implemented
- Improved routing (regex + intent detection function, easy to extend)
- Logging: every query/response is timestamped, logged via the `logging` module, and stored
  in an in-memory `agent_logs` list for auditing
- Four extra tools: **Text Stats**, **DateTime**, **Unit Converter**, and **Sentiment Analyzer**
- Structured JSON output includes `type`, `tool_used`, `result`, `timestamp`, and `query`
- A `help` command in Interactive Mode that lists everything the agent can do


## 📦 Imports & Logging Setup

In [12]:
import re
import json
import logging
from datetime import datetime

# ---- Logging setup ----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("SmartAgent")

# In-memory log of every agent call (bonus: simple audit trail)
agent_logs = []


## 🛠️ TOOL 1: Calculator (safe — no raw `eval`)

In [13]:
import ast
import operator

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed")
    if isinstance(node, ast.BinOp):
        op_type = type(node.op)
        if op_type not in _ALLOWED_OPERATORS:
            raise ValueError(f"Unsupported operator: {op_type.__name__}")
        return _ALLOWED_OPERATORS[op_type](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp):
        op_type = type(node.op)
        if op_type not in _ALLOWED_OPERATORS:
            raise ValueError(f"Unsupported operator: {op_type.__name__}")
        return _ALLOWED_OPERATORS[op_type](_eval_node(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression: str) -> str:
    '''Safely evaluate a mathematical expression (uses ast parsing, not eval()).'''
    expression = expression.strip()
    if not expression:
        return "Error: empty expression"
    try:
        tree = ast.parse(expression, mode="eval")
        result = _eval_node(tree.body)
        return str(result)
    except ZeroDivisionError:
        return "Error: division by zero"
    except Exception as e:
        return f"Error in calculation: {e}"


## 🛠️ TOOL 2: Keyword Extractor (stopword-aware)

In [14]:
STOPWORDS = {
    "this", "that", "with", "from", "have", "about", "which", "there",
    "their", "would", "could", "should", "these", "those", "being",
    "where", "after", "before", "because", "while", "extract",
}


def extract_keywords(text: str, top_n: int = 5) -> list:
    '''Extract simple keywords from text, filtering stopwords/punctuation/duplicates.'''
    try:
        words = re.findall(r"[A-Za-z']+", text)
        keywords, seen = [], set()
        for w in words:
            wl = w.lower()
            if len(wl) > 4 and wl not in STOPWORDS and wl not in seen:
                seen.add(wl)
                keywords.append(wl)
        return keywords[:top_n]
    except Exception:
        return []


## 🛠️ TOOL 3 (bonus): Text Stats

In [15]:
def text_stats(text: str) -> dict:
    '''Return word / character / sentence counts for a piece of text.'''
    try:
        words = text.split()
        chars = len(text)
        sentence_count = len(re.findall(r"[.!?]+", text)) or (1 if text.strip() else 0)
        return {"words": len(words), "characters": chars, "sentences": sentence_count}
    except Exception:
        return {}


## 🛠️ TOOL 4 (bonus): DateTime

In [16]:
def get_datetime(_: str = "") -> str:
    '''Return the current date and time.'''
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


## 🛠️ TOOL 5 (bonus): Unit Converter

In [17]:
_LENGTH_TO_M = {"km": 1000, "m": 1, "cm": 0.01, "mm": 0.001, "mi": 1609.34, "miles": 1609.34, "ft": 0.3048, "feet": 0.3048}
_WEIGHT_TO_KG = {"kg": 1, "g": 0.001, "lb": 0.453592, "lbs": 0.453592, "oz": 0.0283495}

_CONVERT_PATTERN = re.compile(r"([\d.]+)\s*([a-zA-Z]+)\s+to\s+([a-zA-Z]+)", re.IGNORECASE)


def unit_converter(query: str) -> str:
    '''Convert between common length or weight units, e.g. "convert 5 km to miles".'''
    try:
        match = _CONVERT_PATTERN.search(query)
        if not match:
            return "Error: could not parse conversion. Try 'convert 5 km to miles'."

        value, from_unit, to_unit = float(match.group(1)), match.group(2).lower(), match.group(3).lower()

        if from_unit in _LENGTH_TO_M and to_unit in _LENGTH_TO_M:
            result = value * _LENGTH_TO_M[from_unit] / _LENGTH_TO_M[to_unit]
        elif from_unit in _WEIGHT_TO_KG and to_unit in _WEIGHT_TO_KG:
            result = value * _WEIGHT_TO_KG[from_unit] / _WEIGHT_TO_KG[to_unit]
        else:
            return f"Error: unsupported unit pair '{from_unit}' -> '{to_unit}'"

        return f"{value} {from_unit} = {round(result, 4)} {to_unit}"
    except Exception as e:
        return f"Error in conversion: {e}"


## 🛠️ TOOL 6 (bonus): Sentiment Analyzer (lexicon-based)

In [18]:
_POSITIVE_WORDS = {"good", "great", "excellent", "happy", "love", "amazing", "wonderful", "fantastic", "best", "positive"}
_NEGATIVE_WORDS = {"bad", "terrible", "sad", "hate", "awful", "worst", "horrible", "poor", "negative", "angry"}


def sentiment_analyzer(text: str) -> dict:
    '''Simple lexicon-based sentiment scoring of a piece of text.'''
    try:
        words = [w.lower().strip(".,!?") for w in text.split()]
        pos = sum(1 for w in words if w in _POSITIVE_WORDS)
        neg = sum(1 for w in words if w in _NEGATIVE_WORDS)

        if pos > neg:
            label = "positive"
        elif neg > pos:
            label = "negative"
        else:
            label = "neutral"

        return {"label": label, "positive_hits": pos, "negative_hits": neg}
    except Exception:
        return {"label": "neutral", "positive_hits": 0, "negative_hits": 0}


## 🤖 Agent Logic

**Routing rules (bonus: improved routing):**
- Contains "keyword(s)" → Keyword Extractor
- Asks about word/character/sentence counts → Text Stats
- Starts with "convert" (e.g. "convert 5 km to miles") → Unit Converter
- Asks about "sentiment" or "how do you feel about" → Sentiment Analyzer
- Asks for the current time/date → DateTime tool
- Contains "calculate"/"compute", **or** looks like a bare math expression
  (e.g. `"80 + 90"`) → Calculator
- Anything else → General direct response

Every call is logged (timestamp + intent + response) and appended to `agent_logs`.

In [19]:
MATH_EXPRESSION_PATTERN = re.compile(r"^[\d\s\+\-\*/%\.\(\)]+$")


def detect_intent(query: str) -> str:
    q = query.lower().strip()

    if "keyword" in q:
        return "keywords"
    if any(p in q for p in ["word count", "count words", "how many words", "text stats"]):
        return "text_stats"
    if "convert" in q and " to " in q:
        return "unit_conversion"
    if "sentiment" in q or "how do you feel about" in q or "how do i feel about" in q:
        return "sentiment"
    if "time" in q or "date" in q:
        return "datetime"
    if "calculate" in q or "compute" in q or MATH_EXPRESSION_PATTERN.match(q.strip()):
        return "calculation"
    return "general"


def agent(query: str) -> dict:
    '''Single-agent entry point: routes the query, calls the right tool, returns structured JSON.'''
    timestamp = datetime.now().isoformat()
    intent = detect_intent(query)
    logger.info(f"Received query: '{query}' | Detected intent: {intent}")

    response = {
        "query": query,
        "type": intent,
        "tool_used": None,
        "result": None,
        "timestamp": timestamp,
    }

    try:
        if intent == "calculation":
            expression = re.sub(r"calculate|compute", "", query, flags=re.IGNORECASE).strip()
            if not expression:
                response["type"] = "error"
                response["result"] = "No mathematical expression provided."
            else:
                response["tool_used"] = "calculator"
                response["result"] = calculator(expression)

        elif intent == "keywords":
            text = re.sub(r"extract keywords from|keywords", "", query, flags=re.IGNORECASE).strip()
            if not text:
                response["type"] = "error"
                response["result"] = "No text provided for keyword extraction."
            else:
                response["tool_used"] = "keyword_extractor"
                response["result"] = extract_keywords(text)

        elif intent == "text_stats":
            response["tool_used"] = "text_stats"
            response["result"] = text_stats(query)

        elif intent == "datetime":
            response["tool_used"] = "datetime_tool"
            response["result"] = get_datetime()

        elif intent == "unit_conversion":
            response["tool_used"] = "unit_converter"
            response["result"] = unit_converter(query)
            if isinstance(response["result"], str) and response["result"].startswith("Error"):
                response["type"] = "error"

        elif intent == "sentiment":
            text = re.sub(r"sentiment( of| for| analysis)?:?|how do (i|you) feel about", "", query, flags=re.IGNORECASE).strip()
            response["tool_used"] = "sentiment_analyzer"
            response["result"] = sentiment_analyzer(text if text else query)

        else:
            response["type"] = "general"
            response["result"] = f"I understand your question: '{query}'. This doesn't require any tool."

    except Exception as e:
        logger.error(f"Error processing query '{query}': {e}")
        response["type"] = "error"
        response["result"] = str(e)

    agent_logs.append(response)
    logger.info(f"Response: {json.dumps(response)}")
    return response


## 📦 Expected Output Format

```
{
  "query": "...",
  "type": "calculation / keywords / text_stats / datetime / unit_conversion / sentiment / general / error",
  "tool_used": "calculator / keyword_extractor / text_stats / datetime_tool / unit_converter / sentiment_analyzer / null",
  "result": ...,
  "timestamp": "..."
}
```

## 🧪 Test Cases

In [20]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "80 + 90",                       # math expression without the word "calculate"
    "calculate 10 / 0",               # error handling: division by zero
    "calculate",                      # error handling: no expression given
    "How many words are in this sentence right now",
    "What is the current time?",
    "convert 5 km to miles",                          # new: unit converter
    "sentiment: I love this project, it's amazing",    # new: sentiment analyzer
    "sentiment: this traffic is terrible and awful",   # new: sentiment analyzer (negative)
]

for q in queries:
    resp = agent(q)
    print("Query:", q)
    print("Response:", json.dumps(resp, indent=2))
    print("-" * 60)


Query: Calculate 20 + 5
Response: {
  "query": "Calculate 20 + 5",
  "type": "calculation",
  "tool_used": "calculator",
  "result": "25",
  "timestamp": "2026-07-12T17:18:37.598091"
}
------------------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {
  "query": "Extract keywords from Artificial Intelligence is transforming industries",
  "type": "keywords",
  "tool_used": "keyword_extractor",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ],
  "timestamp": "2026-07-12T17:18:37.598884"
}
------------------------------------------------------------
Query: What is machine learning?
Response: {
  "query": "What is machine learning?",
  "type": "general",
  "tool_used": null,
  "result": "I understand your question: 'What is machine learning?'. This doesn't require any tool.",
  "timestamp": "2026-07-12T17:18:37.599035"
}
----------------------------------------

## 📊 Log Summary (bonus)

In [21]:
print(f"Total queries processed: {len(agent_logs)}")
type_counts = {}
for entry in agent_logs:
    type_counts[entry["type"]] = type_counts.get(entry["type"], 0) + 1

print("Breakdown by type:")
for t, c in type_counts.items():
    print(f"  {t}: {c}")


Total queries processed: 11
Breakdown by type:
  calculation: 3
  keywords: 1
  general: 1
  error: 1
  text_stats: 1
  datetime: 1
  unit_conversion: 1
  sentiment: 2


## 🎯 Interactive Mode

Run this cell and type queries at the prompt. Type `exit` to stop, or `help` to see what the agent can do.

In [22]:
HELP_TEXT = '''Try things like:
  - Calculate 20 + 5
  - Extract keywords from <some text>
  - How many words are in <some text>
  - convert 5 km to miles
  - sentiment: I love this!
  - What is the current time?
  - exit  (quits interactive mode)'''

while True:
    user_input = input("Enter query (type 'exit' to stop, 'help' for examples): ")
    if user_input.lower() == "exit":
        break
    if user_input.lower() == "help":
        print(HELP_TEXT)
        continue
    print("Response:", json.dumps(agent(user_input), indent=2))


Enter query (type 'exit' to stop, 'help' for examples): 80+80
Response: {
  "query": "80+80",
  "type": "calculation",
  "tool_used": "calculator",
  "result": "160",
  "timestamp": "2026-07-12T17:18:46.957895"
}
Enter query (type 'exit' to stop, 'help' for examples): convert 5 kg to lbs
Response: {
  "query": "convert 5 kg to lbs",
  "type": "unit_conversion",
  "tool_used": "unit_converter",
  "result": "5.0 kg = 11.0231 lbs",
  "timestamp": "2026-07-12T17:19:27.879454"
}
Enter query (type 'exit' to stop, 'help' for examples): sentiment: I am so sad my fav thing broke down
Response: {
  "query": "sentiment: I am so sad my fav thing broke down",
  "type": "sentiment",
  "tool_used": "sentiment_analyzer",
  "result": {
    "label": "negative",
    "positive_hits": 0,
    "negative_hits": 1
  },
  "timestamp": "2026-07-12T17:20:13.032283"
}
Enter query (type 'exit' to stop, 'help' for examples): whats current time
Response: {
  "query": "whats current time",
  "type": "datetime",
  "too